# CS 451/651: Final Project (Fall 2025)

I want to analysis 4 insights
1. Prompt Topic Distribution — What are the most common themes users discuss with LLMs (e.g., coding, writing, reasoning, education)?
2. Content Distribution - What are the top 15 programming languages used by programmers, based on the volume of conversations? What are the top 15 types of programming errors that programmers most frequently fail to resolve on their own, based on error frequency?
3. Conversation Length and Engagement — How long are typical conversations? Are certain topics associated with longer or more interactive dialogues?
4. Model Comparison — Are there stylistic or structural differences between responses from different models (e.g., GPT-4 vs. Claude vs. Gemini)?

Initialiaze LLM and SamplingParams

I have failed several times because of 
1. GPU crashed
2. Out of Memory
3. Engi Core Died
4. json ouput does not make sense
5. One conversation is too long

So I updated my llm to make sure that the LLM will be successful. I finally made it

In [ ]:
import json
import re
import os
import gc
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# ==========================================
# 1. configuration
# ==========================================
MODEL_PATH = "Qwen/Qwen2.5-7B-Instruct-AWQ"
TOTAL_SIZE = 400_000  # target 400k rows of data
BATCH_SIZE = 10_000     #save for 10k rows（in case of crassh）
MAX_LEN = 4096          # upper limit = 4096 cut too long conversation
OUTPUT_FILE = "wildchat_analysis.csv"

# ==========================================
# 2. Initialize vLLM (stable mode + large context)
# ==========================================
print(f"Initialize vLLM engi...")
# AWQ will automatically load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

llm = LLM(
    model=MODEL_PATH, 
    quantization="awq", 
    gpu_memory_utilization=0.7, # give more GPU memory
    max_model_len=MAX_LEN,      # max length = 4096
    trust_remote_code=True,
    enforce_eager=True          # Prevent Engine Core Died
)

sampling_params = SamplingParams(temperature=0, max_tokens=120)

LLM-based annotation and stream load data

I tried to load all the data into df and failed because I dont have enough RAM

In [ ]:
# ==========================================
# 3. Prepare System Prompt
# ==========================================
SYSTEM_PROMPT = """
Analyze the user's prompt and extract information into a JSON object:
1. "cat": Category [Coding, Creative Writing, Math, Roleplay, General Knowledge, Translation, Data Analysis, Reasoning, Advice, Other].
2. "lang": If Coding, programming language (e.g., "Python"). Else "None".
3. "err": If Coding, error message (e.g., "SyntaxError"). Else "None".
Output JSON ONLY.
"""


In [5]:
# ==========================================
# 4. Stream process (by batch)
# ==========================================
print("laoding dataset streammingly")
ds = load_dataset("allenai/WildChat-4.8M", split="train", streaming=True)
iterator = iter(ds)

laoding dataset streammingly


In case I failed in the middle of the way somehow, save some data into the document and when I rerun my code, I don't need to start from the beginning

In [ ]:

# check if the document exists, support resuming interrupted downloads
processed_count = 0
if os.path.exists(OUTPUT_FILE):
    try:
        df_exist = pd.read_csv(OUTPUT_FILE)
        processed_count = len(df_exist)
        print(f"found process, already processed {processed_count} rows, will skip")
        # Skip processed data (skipping is slower for streaming data, but this is the safest approach)
        for _ in tqdm(range(processed_count), desc="Skipping"):
            next(iterator, None)
    except:
        pass

Using chat template to get all the information I need, like what theme, what language, what error and save all the information in one file

In [ ]:
# process bar
pbar = tqdm(total=TOTAL_SIZE, initial=processed_count)

while processed_count < TOTAL_SIZE:
    # --- 1. collect one Batch ---
    batch_prompts = []     # original Prompt
    batch_inputs = []      # converted Chat Template
    
    while len(batch_inputs) < BATCH_SIZE:
        try:
            row = next(iterator)
            conv = row.get("conversation", [])
            if len(conv) >= 1 and conv[0]['role'] == 'user':
                prompt = conv[0]['content'] or ""
                
                # basic cleaning
                if row.get("language") == "English" and 5 < len(prompt) < 10000:
                    # build Chat Template
                    messages = [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": prompt}
                    ]
                    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                    
                    # calculate token length
                    # drop too long token
                    token_ids = tokenizer.encode(text_input)
                    if len(token_ids) < MAX_LEN - 50: 
                        batch_inputs.append(text_input)
                        batch_prompts.append(prompt)
                    else:
                        # too long, drop it
                        pass
                        
        except StopIteration:
            break
    
    if not batch_inputs:
        break

    # --- 2. batch processing ---
    # print(f"Processing Batch: {len(batch_inputs)} 条...")
    outputs = llm.generate(batch_inputs, sampling_params, use_tqdm=False)

    # --- 3. analysis result ---
    results = []
    for i, output in enumerate(outputs):
        raw_text = output.outputs[0].text.strip()
        # clean JSON
        clean_text = raw_text.replace("```json", "").replace("```", "").strip()
        
        cat, lang, err = "Error", "None", "None"
        try:
            data = json.loads(clean_text)
            cat = data.get("cat", "Other")
            lang = data.get("lang", "None")
            err = data.get("err", "None")
        except:
            pass
        
        results.append({
            "prompt_preview": batch_prompts[i][:100],
            "category": cat,
            "language": lang,
            "error": err
        })

    # --- 4. write file (Append mode) ---
    df_batch = pd.DataFrame(results)
    # if file does not exist, write header, else, do not write
    header = not os.path.exists(OUTPUT_FILE)
    df_batch.to_csv(OUTPUT_FILE, mode='a', index=False, header=header)
    
    # update process rate
    current_batch_len = len(results)
    processed_count += current_batch_len
    pbar.update(current_batch_len)
    
    # Forced garbage collection to prevent memory leaks
    del batch_prompts, batch_inputs, outputs, df_batch, results
    gc.collect()

pbar.close()
print(f"\nAll dont！Result saved: {OUTPUT_FILE}")

I already have the wildchat_analysis csv file and I want to load all the data into dataframe.
I will find all the categories, lauguages and errors frequencies. 

In [ ]:
import pandas as pd

print("reading data (around 400000 rows, please wait)...")
df = pd.read_csv("wildchat_analysis.csv")

# simeple clean
df['category'] = df['category'].fillna('Other').astype(str).str.title().str.strip()
df['language'] = df['language'].fillna('None').astype(str).str.title().str.strip()
df['error'] = df['error'].fillna('None').astype(str).str.strip()

print(f"loading data done，total {len(df)} rows。")

# ==========================================
# 1. What does users ask frequently?(Category distribution)
# ==========================================
top_cats = df['category'].value_counts().head(15)
print("\nTop 15 ask category：")
print(top_cats)

# ==========================================
# 2. What languages does programmers use?(language distribution)？
# ==========================================
# filter invalid values
code_df = df[~df['language'].isin(['None', 'Other', 'Unknown', 'Null', 'Nan'])]
top_langs = code_df['language'].value_counts().head(15)

print("\nTop 15 languages：")
print(top_langs)

# ==========================================
# 3. What errors does programmers have？(error name distribution)
# ==========================================
# filter invalid and too short value
err_df = df[
    (~df['error'].isin(['None', 'Other', 'Unknown'])) & 
    (df['error'].str.len() > 3)
]
top_errors = err_df['error'].value_counts().head(15)

print("\nTop 15 Error message：")
print(top_errors)

# ==========================================
# 4. save the reuslt
# ==========================================
with open("final_report.txt", "w", encoding="utf-8") as f:
    f.write("=== WildChat 400k Analysis Report ===\n\n")
    f.write("[Top Categories]\n")
    f.write(top_cats.to_string() + "\n\n")
    f.write("[Top Languages]\n")
    f.write(top_langs.to_string() + "\n\n")
    f.write("[Top Errors]\n")
    f.write(top_errors.to_string() + "\n")

print("\n detailed final report has been saved as final_report.txt")

Top 15 ask category：

category
| Category            | Count |
|---------------------|-------|
| Creative Writing    | 82432 |
| Error               | 81613 |
| Roleplay            | 74593 |
| General Knowledge   | 52852 |
| Coding              | 39774 |
| Reasoning           | 17980 |
| Translation         | 10225 |
| Advice              | 8789  |
| Math                | 7285  |
| Data Analysis       | 5747  |
| Other               | 4851  |
| History             | 1056  |
| Travel              | 865   |
| Chemistry           | 809   |
| Science             | 805   |

Name: count, dtype: int64

Top 15 ask languages:

language
| Language     | Count |
|--------------|-------|
| Python       | 12786 |
| Javascript   | 5355  |
| English      | 4415  |
| Java         | 2839  |
| C#           | 2225  |
| C++          | 2201  |
| C            | 1189  |
| Vba          | 817   |
| Sql          | 576   |
| Lua          | 564   |
| Matlab       | 540   |
| Php          | 519   |
| Html         | 441   |
| Typescript   | 439   |
| R            | 418   |

Name: count, dtype: int64

Top 15 Error message：

error
| Error Message                                      | Count |
|---------------------------------------------------|-------|
| SyntaxError                                       | 467   |
| SyntaxError: invalid syntax                       | 89    |
| SyntaxError: unexpected EOF while parsing         | 24    |
| IndexError: list index out of range                | 22    |
| Segmentation fault                                | 14    |
| TODO                                              | 13    |
| undefined                                         | 10    |
| SyntaxError: EOL while scanning string literal     | 10    |
| java.lang.NullPointerException                    | 9     |
| IndentationError: expected an indented block       | 9     |
| TypeError: 'NoneType' object is not subscriptable  | 9     |
| SyntaxError: invalid character ‘’ in string        | 9     |
| SyntaxError: Unexpected end of input               | 8     |
| NameError: name 'name' is not defined              | 8     |
| wrong signals                                     | 8     |

Name: count, dtype: int64


Here we can see the three tables alings with our expectation. The only small mistake is the language shall not contain English but it still counts, I guess it is because many users asks to translate anything into English.

In [ ]:
rows = []

for row in tqdm(ds, total=400_000):
    conv = row.get("conversation", [])
    if len(conv) < 2:
        continue

    num_turns = len(conv)
    user_turns = sum(1 for m in conv if m["role"] == "user")
    assistant_turns = sum(1 for m in conv if m["role"] == "assistant")

    total_chars = sum(len(m.get("content", "")) for m in conv)

    rows.append({
        "num_turns": num_turns,
        "user_turns": user_turns,
        "assistant_turns": assistant_turns,
        "total_chars": total_chars,
        "is_multi_turn": num_turns >= 4
    })

df_len = pd.DataFrame(rows)


3199860it [1:25:40, 622.43it/s]                          


In [12]:
df_len.describe(percentiles=[0.5, 0.75, 0.9, 0.95])


,num_turns,user_turns,assistant_turns,total_chars
count,3.199860e+06,3.199860e+06,3.199860e+06,3.199860e+06
mean,3.229630e+00,1.614815e+00,1.614815e+00,9.358622e+03
std,5.081712e+00,2.540856e+00,2.540856e+00,2.682375e+04
min,2.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
50%,2.000000e+00,1.000000e+00,1.000000e+00,2.799000e+03
75%,2.000000e+00,1.000000e+00,1.000000e+00,6.333000e+03
90%,6.000000e+00,3.000000e+00,3.000000e+00,2.209100e+04
95%,1.000000e+01,5.000000e+00,5.000000e+00,4.546600e+04
max,1.000000e+03,5.000000e+02,5.000000e+02,1.091961e+06


| Statistic | Num Turns | User Turns | Assistant Turns | Total Chars |
|----------:|----------:|-----------:|----------------:|------------:|
| count     | 3,199,860 | 3,199,860  | 3,199,860       | 3,199,860   |
| mean      | 3.229630  | 1.614815   | 1.614815        | 9,358.622   |
| std       | 5.081712  | 2.540856   | 2.540856        | 26,823.75   |
| min       | 2         | 1          | 1               | 1           |
| 50%       | 2         | 1          | 1               | 2,799       |
| 75%       | 2         | 1          | 1               | 6,333       |
| 90%       | 6         | 3          | 3               | 22,091      |
| 95%       | 10        | 5          | 5               | 45,466      |
| max       | 1,000     | 500        | 500             | 1,091,961   |

Most conversations are short, with a median of two total turns (one user turn and one assistant turn). However, a small fraction of conversations are significantly longer, as reflected by the heavy-tailed distribution in both turn counts and total character length.

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from tqdm import tqdm
import time


SAMPLE_LIMIT = 100000  
MIN_TEXT_LEN = 3       
SAVE_DOCS_PATH = "wildchat_100k_data_with_topics.csv" 


docs = []        
models = []     

scanned_count = 0
start_time = time.time()

pbar = tqdm(total=SAMPLE_LIMIT, desc="get", unit="rows")

for record in ds:
    scanned_count += 1
    
    if scanned_count % 10000 == 0:
        elapsed = time.time() - start_time
        speed = scanned_count / elapsed
        pbar.write(f"--- scanned: {scanned_count} | spped: {speed:.0f} item/s ---")

    if record.get('language') == 'English' and record['conversation']:
        first_msg = record['conversation'][0]
        if first_msg['role'] == 'user' and first_msg['content']:
            text = first_msg['content']
            model_name = record.get('model', 'unknown')
            
            # simple clean
            if len(str(text).split()) > MIN_TEXT_LEN: 
                docs.append(text)
                models.append(model_name)
                pbar.update(1)
                
                if len(docs) >= SAMPLE_LIMIT:
                    break

pbar.close()
embedding_model = SentenceTransformer("all-mpnet-base-v2", device="cuda")


custom_stop_words = list(CountVectorizer(stop_words="english").get_stop_words())
chat_stopwords = ['please', 'help', 'write', 'create', 'make', 'chatgpt', 'openai', 'code']
custom_stop_words.extend(chat_stopwords)


vectorizer_model = CountVectorizer(
    stop_words=custom_stop_words, 
    ngram_range=(1, 2), 
    min_df=10 
)


topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="english", 
    calculate_probabilities=False, 
    verbose=True,
    min_topic_size=100, 
    nr_topics="auto"    
)

topics, probs = topic_model.fit_transform(docs)

freq = topic_model.get_topic_info()
freq.to_csv("wildchat_100k_topics_list.csv", index=False)

df_full = pd.DataFrame({
    'text': docs,
    'model': models,
    'topic': topics
})

topic_map = freq[['Topic', 'Name']].set_index('Topic')['Name'].to_dict()
df_full['topic_name'] = df_full['topic'].map(topic_map)

df_full['length'] = df_full['text'].apply(lambda x: len(str(x).split()))

df_full.to_csv(SAVE_DOCS_PATH, index=False)


In [29]:
import pandas as pd

df = pd.read_csv("wildchat_100k_data_with_topics.csv")

df["text_char_len"] = df["text"].astype(str).str.len()
df["text_word_len"] = df["text"].astype(str).str.split().str.len()

avg_text_len_by_topic = (
    df
    .groupby("topic_name")["text_word_len"]
    .mean()
    .sort_values(ascending=False)
)

top10 = avg_text_len_by_topic.head(10)
print(top10)



topic_name
89_infinity_secrets_truth_relationships     1457.592920
92_err_musician_res_city                    1409.722222
68_jewelry_women_ring_chain                 1234.938356
33_args_file_gpt_memory                     1126.507645
45_void_device_vector_include                836.217391
78_baby_room_just want_did                   635.566667
9_yuri_okay_ll_just                          602.615840
77_dialogue_illegal_fictional_convincing     500.975207
27_nail_hair_diy_art                         500.233115
41_android_view_products_import              480.077236
Name: text_word_len, dtype: float64


| Theme | Average Words per Text |
|------|------------------------|
| Infinity, Secrets, Truth & Relationships | 1457.6 |
| Musician Errors & City Context | 1409.7 |
| Jewelry, Women, Rings & Chains | 1234.9 |
| GPT Arguments, File Handling & Memory | 1126.5 |
| Device, Vectors & Include Statements | 836.2 |
| Baby Room & Personal Life Questions | 635.6 |
| Informal Dialogue & Casual Queries | 602.6 |
| Illegal or Fictional Dialogue Scenarios | 501.0 |
| Nail, Hair & DIY Art | 500.2 |
| Android Views, Products & Imports | 480.1 |


We analyze conversation length by measuring the average number of words in user texts for each topic.
The table above reports the top 10 topics with the longest average text length.

Several clear patterns emerge from this analysis.

First, topics related to abstract concepts and personal narratives, such as Infinity, Secrets, Truth & Relationships, exhibit the longest average text length. These conversations often involve storytelling, emotional reflection, or philosophical discussion, which naturally leads to long, detailed user inputs.

Second, creative and lifestyle-related topics, including Jewelry Design, Nail and Hair DIY Art, and Personal Life Questions, also show relatively long texts. These topics frequently require descriptive language and contextual background, resulting in higher word counts.

Third, technical topics, such as GPT Arguments, File Handling & Memory and Android Views and Imports, demonstrate substantial text length as well. This suggests that technical problem-solving often involves detailed explanations, code context, and iterative clarification, leading to longer user messages.

Overall, the results indicate that conversation length is strongly topic-dependent. Topics involving personal expression, creative description, or complex technical reasoning tend to generate significantly longer user texts than factual or transactional queries. This supports the conclusion that engagement and interaction depth vary systematically across topics.

Now I am gonna to analysis the difference between each model, because the amount of different model usage varies very much and there are 7 models, so I decide to category them into 4 main models. 

I will analysis their conversation lenght, lexical diversity, average word length, contains code or not, contains list or not. 

Generate calssfication rule

put all the data from datasets into their correspoding buckets or skip

In [ ]:
import numpy as np

# ===========================
# 1. configurationand definition
# ===========================
# We sample 5000 rows for each model
TARGET_PER_MODEL = 5000 

# define the bucket used for data storage
data_buckets = {
    "gpt-4o": [],
    "gpt-3.5-turbo": [],
    "o1": [],
    "gpt-4": []
}

# classify all the models into 4 main categrocies
def categorize_model(model_name):
    if not model_name: return "gpt-4" # in case null
    
    if "gpt-4o" in model_name:
        return "gpt-4o"
    elif "gpt-3.5-turbo" in model_name:
        return "gpt-3.5-turbo"
    elif "o1-mini" in model_name or "o1-preview" in model_name:
        return "o1"
    else:
        # all other cases goes into gpt-4
        return "gpt-4"

# ===========================
# 2. stream read and put them into buckets
# ===========================
print("stream reading and sample from classification...")

# show how the work goes
pbar = tqdm(unit="row", desc="speeding")

for record in ds:
    # 1.get the original name
    raw_model = record.get('model', '')
    
    # 2. apply the categorization rule
    category = categorize_model(raw_model)
    
    # 3.check if the bucket is full or not
    if len(data_buckets[category]) < TARGET_PER_MODEL:
        if record.get('language') == 'English' and record['conversation']:
            first_msg = record['conversation'][0]
            if first_msg['role'] == 'user' and first_msg['content']:
                # store the content only
                data_buckets[category].append(first_msg['content'])
    
    pbar.update(1)
    
    # 4. check if all buckets are full
    all_full = all(len(bucket) >= TARGET_PER_MODEL for bucket in data_buckets.values())
    if all_full:
        print("All buckets are full, stop scanning")
        break

pbar.close()

Do the sample and count for each character

In [ ]:
# ===========================
# 3. Downsampling
# ===========================


balanced_data = []
for model, data in data_buckets.items():
    # get 5000 rows
    sample = data[:5000]
    for text in sample:
        balanced_data.append({"model": model, "text": text})

df = pd.DataFrame(balanced_data)

# ===========================
# 4. stylistic character computation
# ===========================
print("\ncomputing...")

def analyze_style(text):
    text = str(text)
    words = text.split()
    word_count = len(words)
    
    # 1. length
    char_count = len(text)
    
    # 2. diversity
    if word_count > 0:
        unique_words = len(set(words))
        ttr = unique_words / word_count
        avg_word_len = char_count / word_count
    else:
        ttr = 0
        avg_word_len = 0
        
    # 3. if contains code
    has_code = 1 if "```" in text else 0
    
    # 4. if contains list
    has_list = 1 if re.search(r'(?m)^(\s*[-*]|\s*\d+\.) ', text) else 0
    
    return pd.Series([word_count, ttr, avg_word_len, has_code, has_list])

# apply the function
df[['word_count', 'lexical_diversity', 'avg_word_len', 'has_code', 'has_list']] = df['text'].apply(analyze_style)

get the result and print using a table and find top 1

In [ ]:
# ===========================
# 5. get the result and print
# ===========================

# get average
result = df.groupby('model')[['word_count', 'lexical_diversity', 'avg_word_len', 'has_code', 'has_list']].mean()

print("\n====== stylistic or structural differences from different models (average) ======")
print(result)

# save all the record
df.to_csv("model_style_comparison.csv", index=False)
print("\ndetailed data analysis saved as 'model_style_comparison.csv'")

# ===========================
# 6. print the result
# ===========================
print("\n====== simple result ======")
# which one says most
longest_model = result['word_count'].idxmax()
print(f"1. model with longest conversation is : {longest_model} ({result.loc[longest_model, 'word_count']:.1f} words)")

# which one codes most
coder_model = result['has_code'].idxmax()
print(f"2. modes always contains code is : {coder_model} (contribute to {result.loc[coder_model, 'has_code']*100:.1f}%)")

# which one dont have enough words to say
repetitive_model = result['lexical_diversity'].idxmin()
print(f"3. model most repetively uses word is : {repetitive_model}")


Here is the output :Comparison of response characteristics across different language models, including verbosity, lexical diversity, and structural features.

====== stylistic or structural differences from different models (average) ======

| Model          | Word Count | Lexical Diversity | Avg Word Length | Has Code | Has List |
|----------------|------------|-------------------|-----------------|----------|----------|
| gpt-3.5-turbo  | 112.0154   | 0.850464          | 6.755645        | 0.0036   | 0.0416   |
| gpt-4          | 266.6674   | 0.791679          | 6.975801        | 0.0026   | 0.0700   |
| gpt-4o         | 445.7236   | 0.798059          | 6.405032        | 0.0090   | 0.0568   |
| o1             | 306.8714   | 0.798521          | 6.489153        | 0.0280   | 0.0824   |


detailed data analysis saved as 'model_style_comparison.csv



====== simple result ======

1. model with longest conversation is : gpt-4o (445.7 words)

2. modes always contains code is : o1 (contribute to 2.8%)

3. model most repetively uses word is : gpt-4
